# Validação 18 — API HTTP assíncrona

## Goal

Validar o contrato HTTP que expõe o serviço multiartigo para uma futura interface. A API deve aceitar uma alegação, devolver imediatamente um identificador, processar a análise fora da requisição e permitir a consulta posterior do estado e do resultado.

Esta etapa reutiliza somente o padrão arquitetural observado no projeto de referência. O núcleo científico continua sendo o `MultiArticleAnalysisService` validado na etapa 17, sem Qdrant, MinIO ou n8n.

## Setup

O teste usa o cliente HTTP do Flask, mas não abre uma porta de rede. Dois artigos controlados atravessam o serviço científico real, o que mantém o notebook rápido e determinístico enquanto valida integração, serialização e estados assíncronos.

In [1]:
import sys
import time
from pathlib import Path

from IPython.display import JSON, Markdown, display

project_root = Path.cwd()
if not (project_root / 'src').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from fatofake import (
    ArticleContent,
    ArticleEvidenceBundle,
    ArticleQualityReport,
    EvidenceAssessment,
    EvidenceStatement,
    MultiArticleAnalysisConfig,
    MultiArticleAnalysisService,
    Publication,
    QualityLevel,
    RelationLabel,
    RelationProbabilities,
    StudyDesign,
)
from fatofake.api import AnalysisJobService, create_app


def display_table(rows, columns):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join('---' for _ in columns) + ' |'
    body = [
        '| ' + ' | '.join(str(row.get(column, '')).replace('|', r'\|') for column in columns) + ' |'
        for row in rows
    ]
    display(Markdown('\n'.join([header, separator, *body])))

## Steps

### 1. Preparar um pipeline científico controlado

Os conectores são substituídos por dados determinísticos, mas a agregação entre artigos, a síntese e a geração do relatório são as implementações reais do projeto.

In [2]:
claim = 'Beber café pode alterar o risco de câncer de próstata.'
query = 'coffee[Title] AND prostate[Title]'


def publication(pmid, title):
    return Publication(
        pmid=pmid,
        title=title,
        authors=('Pesquisador A', 'Pesquisador B'),
        journal='Periódico controlado',
        publication_date='2024',
        doi=f'10.1000/{pmid}',
        url=f'https://pubmed.ncbi.nlm.nih.gov/{pmid}/',
        matched_queries=(query,),
    )


def bundle(item):
    statement = EvidenceStatement(
        statement_id=f'statement:{item.pmid}',
        text='Coffee consumption was associated with prostate cancer risk.',
        extraction_score=1.0,
        matched_claim_terms=('cancer', 'risco'),
        hybrid_rank=1,
        sentence_index=1,
        chunk_id=f'chunk:{item.pmid}',
        pmid=item.pmid,
        pmcid=None,
        doi=item.doi,
        section='Abstract',
        source_url=item.url,
    )
    assessment = EvidenceAssessment(
        pair_id=f'pair:{item.pmid}',
        relation=RelationLabel.SUPPORTS,
        model_relation=RelationLabel.SUPPORTS,
        confidence=0.80,
        margin=0.70,
        probabilities=RelationProbabilities(0.80, 0.10, 0.10),
        rationale='Resultado controlado para validar o contrato HTTP.',
        model_name='controlled-nli',
        claim=claim,
        evidence=statement,
    )
    content = ArticleContent(
        pmid=item.pmid,
        pmcid=None,
        doi=item.doi,
        abstract=statement.text,
        full_text=None,
        sections=(),
        access_level='ABSTRACT_ONLY',
        pubmed_url=item.url,
        pmc_url=None,
    )
    quality = ArticleQualityReport(
        pmid=item.pmid,
        doi=item.doi,
        study_design=StudyDesign.OBSERVATIONAL,
        quality_level=QualityLevel.UNCLEAR,
        checks=(),
        datasets=(),
        trial_registrations=(),
        rationale='Perfil controlado e conservador.',
    )
    return ArticleEvidenceBundle(item, content, (assessment,), quality)


publications = (
    publication('1001', 'Coffee consumption and prostate cancer risk: cohort study.'),
    publication('1002', 'Coffee intake and prostate cancer: independent cohort.'),
)


class ControlledPlanner:
    def generate_queries(self, _claim):
        return [query]


class ControlledPubMedClient:
    def search_ids(self, _query, *, max_results):
        return len(publications), tuple(item.pmid for item in publications[:max_results])

    def fetch_summaries(self, identifiers, _matched_queries):
        by_id = {item.pmid: item for item in publications}
        return tuple(by_id[pmid] for pmid in identifiers)


class ControlledProcessor:
    def __init__(self):
        self.items = {item.pmid: bundle(item) for item in publications}

    def process(self, item, _claim):
        return self.items[item.pmid]


scientific_service = MultiArticleAnalysisService(
    query_planner=ControlledPlanner(),
    pubmed_client=ControlledPubMedClient(),
    article_processor=ControlledProcessor(),
    config=MultiArticleAnalysisConfig(
        max_results_per_query=2,
        target_articles=2,
        minimum_successful_articles=2,
    ),
)
print('Pipeline controlado preparado com dois artigos independentes.')

Pipeline controlado preparado com dois artigos independentes.


### 2. Criar a API e iniciar uma análise

Um pequeno atraso controlado permite observar o estado intermediário. Em produção, essa duração vem naturalmente das consultas e dos modelos científicos.

In [3]:
class DelayedRunner:
    def __init__(self, service, delay=0.15):
        self.service = service
        self.delay = delay

    def analyze(self, claim, article_reference=None):
        time.sleep(self.delay)
        return self.service.analyze(claim, article_reference)


job_service = AnalysisJobService(DelayedRunner(scientific_service), max_workers=1)
app = create_app(job_service)
client = app.test_client()

created_response = client.post(
    '/api/v1/analyses',
    json={'claim': claim, 'article_reference': None},
)
created = created_response.get_json()

display(Markdown('#### Resposta da criação'))
display(JSON(created, expanded=True))
print('HTTP:', created_response.status_code)
print('Location:', created_response.headers.get('Location'))

#### Resposta da criação

<IPython.core.display.JSON object>

HTTP: 202
Location: /api/v1/analyses/3d98650d-7891-4eaf-ad9f-cf221f252052


### 3. Consultar progresso até a conclusão

O cliente usa a URL devolvida pela própria API. Essa separação evita manter uma requisição HTTP aberta durante todo o pipeline.

In [4]:
observed_states = []
retry_after_seen = None
for _ in range(100):
    status_response = client.get(created['status_url'])
    status_payload = status_response.get_json()
    state = (status_payload['status'], status_payload['progress'])
    if not observed_states or observed_states[-1] != state:
        observed_states.append(state)
    if status_payload['status'] in {'QUEUED', 'RUNNING'}:
        retry_after_seen = status_response.headers.get('Retry-After')
    if status_payload['status'] in {'SUCCEEDED', 'FAILED'}:
        break
    time.sleep(0.01)
else:
    raise TimeoutError('A análise controlada não terminou no tempo esperado.')

display_table(
    [
        {'ordem': index, 'estado': status, 'progresso': f'{progress}%'}
        for index, (status, progress) in enumerate(observed_states, start=1)
    ],
    ['ordem', 'estado', 'progresso'],
)
print(f'Retry-After observado durante processamento: {retry_after_seen} segundo.')

| ordem | estado | progresso |
| --- | --- | --- |
| 1 | RUNNING | 10% |
| 2 | SUCCEEDED | 100% |

Retry-After observado durante processamento: 1 segundo.


### 4. Inspecionar o resultado JSON

O retorno mantém a síntese conservadora, os artigos independentes e as fontes necessárias para auditoria pela interface.

In [5]:
result = status_payload['result']
summary_rows = [{
    'conclusão': result['report']['conclusion'],
    'direção': result['synthesis']['direction'],
    'força': result['synthesis']['strength'],
    'artigos': result['synthesis']['article_count'],
    'avaliações metodológicas': len(result['report']['methodology']),
    'fontes': len(result['report']['sources']),
}]
display_table(
    summary_rows,
    ['conclusão', 'direção', 'força', 'artigos', 'avaliações metodológicas', 'fontes'],
)

display(Markdown('#### Artigos devolvidos pela API'))
display_table(
    [
        {
            'PMID': article['pmid'],
            'título': article['title'],
            'qualidade': article['quality']['level'],
            'evidências': len(article['assessments']),
        }
        for article in result['articles']
    ],
    ['PMID', 'título', 'qualidade', 'evidências'],
)

| conclusão | direção | força | artigos | avaliações metodológicas | fontes |
| --- | --- | --- | --- | --- | --- |
| COMPATIBLE_WITH_EVIDENCE | SUPPORTS | LOW | 2 | 2 | 4 |

#### Artigos devolvidos pela API

| PMID | título | qualidade | evidências |
| --- | --- | --- | --- |
| 1001 | Coffee consumption and prostate cancer risk: cohort study. | UNCLEAR | 1 |
| 1002 | Coffee intake and prostate cancer: independent cohort. | UNCLEAR | 1 |

### 5. Verificar erros de contrato

Entradas inválidas são recusadas antes do agendamento e identificadores desconhecidos não expõem detalhes internos.

In [6]:
invalid_response = client.post('/api/v1/analyses', json={'claim': 'curta'})
missing_response = client.get('/api/v1/analyses/nao-existe')

error_rows = [
    {
        'caso': 'alegação inválida',
        'HTTP': invalid_response.status_code,
        'código': invalid_response.get_json()['error']['code'],
    },
    {
        'caso': 'análise inexistente',
        'HTTP': missing_response.status_code,
        'código': missing_response.get_json()['error']['code'],
    },
]
display_table(error_rows, ['caso', 'HTTP', 'código'])

| caso | HTTP | código |
| --- | --- | --- |
| alegação inválida | 422 | INVALID_INPUT |
| análise inexistente | 404 | ANALYSIS_NOT_FOUND |

## Checks

As verificações abaixo cobrem o contrato mínimo necessário para a interface.

In [7]:
checks = {
    'criação retorna HTTP 202': created_response.status_code == 202,
    'resposta inclui Location': created_response.headers.get('Location') == created['status_url'],
    'processamento passou por RUNNING': any(status == 'RUNNING' for status, _ in observed_states),
    'análise termina com sucesso': status_payload['status'] == 'SUCCEEDED',
    'progresso final é 100%': status_payload['progress'] == 100,
    'Retry-After orienta nova consulta': retry_after_seen == '1',
    'dois artigos independentes': result['synthesis']['article_count'] == 2,
    'metodologia cobre os dois artigos': len(result['report']['methodology']) == 2,
    'fontes cobrem os dois PMIDs': {s['pmid'] for s in result['report']['sources'] if s['pmid']} == {'1001', '1002'},
    'entrada inválida retorna 422': invalid_response.status_code == 422,
    'identificador desconhecido retorna 404': missing_response.status_code == 404,
    'CORS não foi aberto globalmente': 'Access-Control-Allow-Origin' not in created_response.headers,
}

display_table(
    [{'verificação': name, 'resultado': 'OK' if passed else 'FALHOU'} for name, passed in checks.items()],
    ['verificação', 'resultado'],
)
assert all(checks.values())
print('Todas as verificações passaram.')
job_service.close()

| verificação | resultado |
| --- | --- |
| criação retorna HTTP 202 | OK |
| resposta inclui Location | OK |
| processamento passou por RUNNING | OK |
| análise termina com sucesso | OK |
| progresso final é 100% | OK |
| Retry-After orienta nova consulta | OK |
| dois artigos independentes | OK |
| metodologia cobre os dois artigos | OK |
| fontes cobrem os dois PMIDs | OK |
| entrada inválida retorna 422 | OK |
| identificador desconhecido retorna 404 | OK |
| CORS não foi aberto globalmente | OK |

Todas as verificações passaram.


## Next Steps

A API já separa o tempo de processamento do ciclo da requisição e entrega um JSON auditável. O armazenamento de trabalhos ainda é em memória; portanto, reiniciar o processo elimina o histórico e múltiplas instâncias não compartilham estado.

A próxima etapa é construir uma interface mínima que envie a alegação, consulte `status_url` periodicamente e apresente conclusão, força, limitações e fontes. Persistência em PostgreSQL e filas externas devem entrar somente quando o MVP precisar sobreviver a reinícios ou escalar horizontalmente.